In [16]:
import numpy as np 
# import sklearn as sk
import matplotlib.pyplot as plt
import pandas as pd


In [17]:
def pmatrix(a, digit : int =3, T : bool =False):
    """Returns a LaTeX bmatrix
    :a: numpy array
    :returns: LaTeX bmatrix as a string
    """
    if len(a.shape) > 2:
        raise ValueError('bmatrix can at most display two dimensions')
    lines = str(np.round(a, digit)).replace('[', '').replace(']', '').splitlines()
    rv = [r'\begin{pmatrix}']
    if not T:
        rv += ['  ' + ' & '.join(l.split()) + r'\\' for l in lines]
    else:
        rv += ['  ' + '\\\\ '.join(l.split()) for l in lines]
    rv +=  [r'\end{pmatrix}']
    return '\n'.join(rv)

In [18]:
np.set_printoptions(precision=3, formatter={'float_kind': '{:0.3f}'.format})

In [19]:
x1 = np.array([-1.000, -0.709, -0.418, -0.127, 0.164, 0.455, 0.746, 1.037, 1.328, 1.619])
x2 = np.array([-2.031, -1.438, -0.833, -0.244, 0.321, 0.917, 1.478, 2.091, 2.653, 3.235])
y = np.array([-7.170, -3.754, -1.643, -0.152, 0.002, -0.517, -2.767, -5.400, -9.215, -14.118])
betta = [-1, 1, -3]

In [20]:
F = np.round(np.array([
    x1,
    x2,
    x1 * x2,
]), 4)

In [9]:
print("F = ", pmatrix(F.T,3))
print("y = ", pmatrix(y, 4, T=True))

F =  \begin{pmatrix}
  -1.000 & -2.031 & 2.031\\
  -0.709 & -1.438 & 1.020\\
  -0.418 & -0.833 & 0.348\\
  -0.127 & -0.244 & 0.031\\
  0.164 & 0.321 & 0.053\\
  0.455 & 0.917 & 0.417\\
  0.746 & 1.478 & 1.103\\
  1.037 & 2.091 & 2.168\\
  1.328 & 2.653 & 3.523\\
  1.619 & 3.235 & 5.238\\
\end{pmatrix}
y =  \begin{pmatrix}
  -7.170\\ -3.754\\ -1.643\\ -0.152\\ 0.002\\ -0.517\\ -2.767\\ -5.400\\ -9.215\\ -14.118
\end{pmatrix}


In [21]:
F.T # Матрица регрессоров

array([[-1.000, -2.031, 2.031],
       [-0.709, -1.438, 1.020],
       [-0.418, -0.833, 0.348],
       [-0.127, -0.244, 0.031],
       [0.164, 0.321, 0.053],
       [0.455, 0.917, 0.417],
       [0.746, 1.478, 1.103],
       [1.037, 2.091, 2.168],
       [1.328, 2.653, 3.523],
       [1.619, 3.235, 5.237]])

In [22]:
vrn = F @ y 
print(pmatrix(vrn, T = True)) # вектор системы нормальных уравнений
vrn

\begin{pmatrix}
  -32.456\\ -64.607\\ -140.351
\end{pmatrix}


array([-32.456, -64.607, -140.351])

In [30]:
G = F @ F.T
print(pmatrix(G)) # информационная матрица

\begin{pmatrix}
  7.944 & 15.931 & 13.525\\
  15.931 & 31.951 & 26.965\\
  13.525 & 26.965 & 51.225\\
\end{pmatrix}


In [31]:
C = np.linalg.inv(G)
print(pmatrix(C)) # матрица дисперсий-ковариаций

\begin{pmatrix}
  4240.747 & -2104.548 & -11.820\\
  -2104.548 & 1044.477 & 5.836\\
  -11.820 & 5.836 & 0.068\\
\end{pmatrix}


In [33]:
b = C @ vrn
print("b = ", pmatrix(C), "\\cdot", pmatrix(vrn, T=True), " = \\\\", pmatrix(b, T=False)) # вектор оценок регрессионых
# коэффициентов без учета заданных линейных ограничений

b =  \begin{pmatrix}
  4240.747 & -2104.548 & -11.820\\
  -2104.548 & 1044.477 & 5.836\\
  -11.820 & 5.836 & 0.068\\
\end{pmatrix} \cdot \begin{pmatrix}
  -32.456\\ -64.607\\ -140.351
\end{pmatrix}  = \\ \begin{pmatrix}
  -8.530 & 4.756 & -2.991\\
\end{pmatrix}


In [37]:
def y_pred(b):
    return np.round(b[0]* x1 + b[1] * x2 + b[2] * x1*x2, 3)

In [36]:
print(f'{b[0]:.3f} x_{{1k}} + {b[1]:.3f}x_{{2k}} + {b[2]:.3f} x_{{1k}}x_{{2k}}')

-8.530 x_{1k} + 4.756x_{2k} + -2.991 x_{1k}x_{2k}


In [42]:
S_ost = 1 / (10 - 3) * np.sum(y - y_pred(b))
print(f"$s^2_{{ост}} = \\dfrac{{1}} {{N - n}} \\sum_{{k=1}}^N (y_k - \\hat{{y_k}})^2 = {S_ost:.3f}$")

$s^2_{ост} = \dfrac{1} {N - n} \sum_{k=1}^N (y_k - \hat{y_k})^2 = 0.011$


In [44]:
df = pd.DataFrame({'k': [x for x in range(1, 11)],
               'y_k': y, 
                '\\hat{y_k}': y_pred(b),
                'y-y_k': (y-y_pred(b))**2})

print(df.to_latex(float_format="%.3f", index=False).replace("\\\n", "\\ \hline\n"))

\begin{tabular}{rrrr}
\toprule
k & y_k & \hat{y_k} & y-y_k \\ \hline
\midrule
1 & -7.170 & -7.203 & 0.001 \\ \hline
2 & -3.754 & -3.840 & 0.007 \\ \hline
3 & -1.643 & -1.437 & 0.042 \\ \hline
4 & -0.152 & -0.170 & 0.000 \\ \hline
5 & 0.002 & -0.030 & 0.001 \\ \hline
6 & -0.517 & -0.768 & 0.063 \\ \hline
7 & -2.767 & -2.633 & 0.018 \\ \hline
8 & -5.400 & -5.388 & 0.000 \\ \hline
9 & -9.215 & -9.250 & 0.001 \\ \hline
10 & -14.118 & -14.092 & 0.001 \\ \hline
\bottomrule
\end{tabular}



<>:6: SyntaxWarning: invalid escape sequence '\h'
<>:6: SyntaxWarning: invalid escape sequence '\h'
C:\Users\aveoc\AppData\Local\Temp\ipykernel_8836\4235587934.py:6: SyntaxWarning: invalid escape sequence '\h'
  print(df.to_latex(float_format="%.3f", index=False).replace("\\\n", "\\ \hline\n"))


In [47]:
V_b = C * S_ost # ковариационная матрица оценок V[b]
print("$V[b] = ", pmatrix(V_b))

$V[b] =  \begin{pmatrix}
  46.648 & -23.150 & -0.130\\
  -23.150 & 11.489 & 0.064\\
  -0.130 & 0.064 & 0.001\\
\end{pmatrix}


In [48]:
sb = np.diagonal(V_b)**0.5 # квадратный корень из диаогонального элемента матрицы
tsb = 2.36 * sb # 2.36 из таблицы Распределения Сьюдента при числен степений свободы 7
df2 = pd.DataFrame({
    "$s[b_i]$": sb,
    "t_k s[b]": tsb,
    'b - tsb': b - tsb,
    'b': b,
    'b': b + tsb
})

print(df2.to_latex(float_format="%.3f", index=False).replace("\\\n", "\\ \hline\n"))


\begin{tabular}{rrrr}
\toprule
$s[b_i]$ & t_k s[b] & b - tsb & b \\ \hline
\midrule
6.830 & 16.119 & -24.649 & 7.588 \\ \hline
3.390 & 7.999 & -3.244 & 12.755 \\ \hline
0.027 & 0.065 & -3.056 & -2.926 \\ \hline
\bottomrule
\end{tabular}



<>:11: SyntaxWarning: invalid escape sequence '\h'
<>:11: SyntaxWarning: invalid escape sequence '\h'
C:\Users\aveoc\AppData\Local\Temp\ipykernel_8836\2868283276.py:11: SyntaxWarning: invalid escape sequence '\h'
  print(df2.to_latex(float_format="%.3f", index=False).replace("\\\n", "\\ \hline\n"))


In [16]:
b, betta

(array([-10.015, 6.049, 0.937]), [-2, 2, 1])

In [28]:
f_mean= F.mean(axis=1)
f_mean = np.append(f_mean, y.mean())
l = np.round([np.sum((F[i] - f_mean[i])**2)**0.5 for i in range(3)], 3)
l = np.append(l, np.sum((y - y.mean())**2)**0.5)

print(l)
print(f_mean)

[2.643 5.307 5.084 13.763]
[0.309 0.615 1.593 -4.473]


In [18]:
print(l)

[2.334 4.654 3.365 6.725]


In [19]:
df = pd.DataFrame({'k': [x for x in range(1, 11)],
                    'fk1': F[0],
                    'fk2': F[1],
                    
                    'fk3': F[2],
                    'y': y,
                    'e_k1': np.round((F[0] - f_mean[0]) / l[0], 3),
                    'e_k2': np.round((F[1] - f_mean[1]) / l[1], 3),
                    'e_k3': np.round((F[2] - f_mean[2]) / l[2], 3),
                    'c_k1': np.round((y - f_mean[3]) / l[3], 3)
                   })

print(df.to_latex(float_format="%.3f", index=False).replace("\\\n", "\\ \hline\n"))

\begin{tabular}{rrrrrrrrr}
\toprule
k & fk1 & fk2 & fk3 & y & e_k1 & e_k2 & e_k3 & c_k1 \\ \hline
\midrule
1 & -1.000 & -1.979 & 1.979 & -0.022 & -0.496 & -0.493 & 0.251 & -0.220 \\ \hline
2 & -0.743 & -1.472 & 1.094 & -0.485 & -0.385 & -0.384 & -0.013 & -0.288 \\ \hline
3 & -0.486 & -0.974 & 0.473 & -0.582 & -0.275 & -0.277 & -0.197 & -0.303 \\ \hline
4 & -0.229 & -0.465 & 0.106 & -0.386 & -0.165 & -0.168 & -0.306 & -0.274 \\ \hline
5 & 0.028 & 0.061 & 0.002 & 0.212 & -0.055 & -0.055 & -0.337 & -0.185 \\ \hline
6 & 0.285 & 0.565 & 0.161 & 0.996 & 0.055 & 0.054 & -0.290 & -0.068 \\ \hline
7 & 0.542 & 1.093 & 0.592 & 1.758 & 0.165 & 0.167 & -0.162 & 0.045 \\ \hline
8 & 0.799 & 1.587 & 1.268 & 2.795 & 0.275 & 0.273 & 0.039 & 0.199 \\ \hline
9 & 1.056 & 2.114 & 2.232 & 4.383 & 0.385 & 0.386 & 0.326 & 0.435 \\ \hline
10 & 1.313 & 2.628 & 3.451 & 5.874 & 0.496 & 0.497 & 0.688 & 0.657 \\ \hline
\bottomrule
\end{tabular}



<>:13: SyntaxWarning: invalid escape sequence '\h'
<>:13: SyntaxWarning: invalid escape sequence '\h'
C:\Users\aveoc\AppData\Local\Temp\ipykernel_14076\2912992627.py:13: SyntaxWarning: invalid escape sequence '\h'
  print(df.to_latex(float_format="%.3f", index=False).replace("\\\n", "\\ \hline\n"))


In [ ]:
F_e = df[['e_k1', 'e_k 2', 'e_k3']]
R = F_e.T @ F_e # Матрица кореляций регрессии

In [ ]:
print("Матрица кореляций $R = F_{\\xi}"pmatrix(R))

\begin{pmatrix}
  e_k1 & e_k2 & e_k3\\
  e_k1 & 1.000 & 1.000 & 0.439\\
  e_k2 & 1.000 & 1.000 & 0.441\\
  e_k3 & 0.439 & 0.441 & 1.001\\
\end{pmatrix}


In [27]:
np.linalg.det(R)

np.float64(1.8140085711486417e-05)

In [20]:
print(pd.DataFrame({'k': [x for x in range(1, 11)],
               'y_k': y, 
                '\\hat{y_k}': lin_model(b),
                'y-y_k': std}))

NameError: name 'std' is not defined